# Cohort external-test selection

## Introduction

This notebook selects three whole kidney images for external testing before the training cohort is constructed. The training cohort is every complete source image not selected here; the legacy merged `kidney` directory is excluded.

## Assumptions

The annotation CSV represents the available METASPACE molecular population, and the `x<int>_y<int>` columns in `pixel_intensities.csv` represent pixels. Selection must preserve every molecule label in the remaining training images.

## Notation

For an image $i$, $P_i$ is its pixel count, $A_i$ its annotation-record count and $C_i$ its number of unique `formula|adduct` labels.

## Reproducible metadata scan

### Methodology

The scan reads every annotation CSV and only the header of every intensity CSV. Candidates are inside the 20th--80th percentile for log(1 + P_i), log(1 + A_i) and log(1 + C_i). It enumerates triples, rejects every triple that removes a label from training, and chooses the lowest robust distance to the cohort median.

### Theoretical description

The lossless constraint is: union of labels outside held-out set H equals the union across the whole cohort. Consequently a held-out image cannot be the only training source of any molecular class.

### Implementation description

The reusable implementation is `analysis.autoencoder.experiments.cohort_selection`; it writes canonical CSV tables and provenance. The notebook only loads and presents those artifacts.

### Figure description

No plot is required for this preflight decision; the numeric distribution and selected rows are displayed below.

### Remarks

The scan is metadata-only and does not materialize spectra or an intensity matrix. It is therefore not a throughput benchmark for training.

### Notes


In [ ]:
from pathlib import Path

import pandas as pd

from msi_autoencoder_wrapper.analysis.autoencoder.experiments.cohort_selection import (
    load_cohort_selection_results,
)

RESULTS_DIRECTORY = Path('part_0_1_cohort_selection_results')
results = load_cohort_selection_results(RESULTS_DIRECTORY)
images = results['images']
heldout = results['heldout']
coverage = results['label_coverage']


In [ ]:
display(images[['pixel_count', 'annotation_records', 'unique_label_count']].describe(percentiles=[0.2, 0.5, 0.8]))
display(heldout)
assert (coverage['training_image_count'] > 0).all(), 'A held-out selection removed a training label.'
print(f"Training label coverage: {len(coverage)}/{len(coverage)} labels retained.")
